In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from Base_Modules.Environments import Prison
from Prison_Strategies.Basic_Strategies import *
from Base_Modules.Game_Master import Game_Master
from Base_Modules.Action import Action_History, Prison_Actions, Duel_Matrix
import pandas as pd
from collections import defaultdict

In [3]:
prison = Prison()
actions = prison.Get_Actions()

In [4]:
strategies_list = [
    Random_Strategy(actions=actions),
    Random_Strategy(actions=actions, p_coop=0.1),
    Always_Betray(actions=actions),
    Always_Cooperate(actions=actions),
    Patient_Unforgiving(actions=actions),
    Copycat(actions=actions),
    Periodic(actions=actions, period=1),
]

number_of_strategies = len(strategies_list)

In [5]:
def Get_Index_By_Name(strategies : dict[int, Strategy], name : str) -> int:
    for ix, (id, s) in enumerate(strategies.items()):
        if str(s) == name:
            return id
    for ix, (id, s) in enumerate(strategies.items()):
        if str(s).startswith(name):
            return id
    return -1

In [6]:
strategies = {}

for (i, s) in enumerate(strategies_list):
    strategies[i] = s
    s.Set_ID(i)

In [7]:
number_of_games = 10
total_games_explicit = True
max_action_memory = -1

gm = Game_Master(prison, strategies=strategies, duel_size=2, max_action_memory=max_action_memory)
duel_matrix, rewards = gm.Tournament(number_of_games, Game_Master.Game_Type.All_Vs_All, total_games_explicit=total_games_explicit)
rewards.Sort_Total_Rewards()

In [8]:
print(duel_matrix.Get_Action_History((0, 1)).Get_Action_History())
print(Get_Index_By_Name(strategies, "Periodic"))

{0: [<Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>], 1: [<Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>]}
6


In [9]:
total_rewards = rewards.Get_Total_Rewards()
total_rewards_per_name = {str(strategies[i]):total_rewards[i] for i in total_rewards.keys()}

average_rewards_per_match = {k: (float(v)/number_of_strategies) for k, v in total_rewards_per_name.items()}
average_rewards_per_round = {k: (float(v)/(number_of_strategies*number_of_games)) for k, v in total_rewards_per_name.items()}

duel_rewards = rewards.Get_All_Duel_Rewards()
winner = next(iter(total_rewards))

## Wyniki

In [10]:
average_reward_per_round_df = pd.DataFrame.from_dict(average_rewards_per_round, orient="index", columns=["Average Reward"])
average_reward_per_round_df.index.name = "Strategy Name"
average_reward_per_round_df = average_reward_per_round_df.round(3)
average_reward_per_round_df

,Average Reward
Strategy Name,
Random_Strategy (p_coop=0.1),2.371
Always_Betray,2.057
Patient_Unforgiving (patience=1),1.914
Copycat (1st:Cooperate),1.886
Periodic (period=1),1.729
Random_Strategy (p_coop=0.5),1.514
Always_Cooperate,1.200


In [17]:
from Base_Modules.Nemesis import Nemesis_Best_Enemy_Score, Nemesis_Worst_Score

print(duel_rewards)
print("\n")
nemesis = Nemesis_Best_Enemy_Score.Get_Nemesis(duel_rewards=duel_rewards)
print(nemesis)
print(Nemesis_Best_Enemy_Score.Translate_Nemesis_To_Strategy_Names(strategies, nemesis))

{(0, 1): {0: 4, 1: 44}, (0, 2): {0: 6, 2: 26}, (0, 3): {0: 44, 3: 9}, (0, 4): {0: 9, 4: 29}, (0, 5): {0: 25, 5: 25}, (0, 6): {0: 18, 6: 28}, (1, 2): {1: 10, 2: 10}, (1, 3): {1: 50, 3: 0}, (1, 4): {1: 14, 4: 9}, (1, 5): {1: 20, 5: 15}, (1, 6): {1: 28, 6: 8}, (2, 3): {2: 50, 3: 0}, (2, 4): {2: 14, 4: 9}, (2, 5): {2: 14, 5: 9}, (2, 6): {2: 30, 6: 5}, (3, 4): {3: 30, 4: 30}, (3, 5): {3: 30, 5: 30}, (3, 6): {3: 15, 6: 40}, (4, 5): {4: 30, 5: 30}, (4, 6): {4: 27, 6: 12}, (5, 6): {5: 23, 6: 28}}


{0: (1, {0: 4, 1: 44}), 1: (5, {1: 20, 5: 15}), 2: (1, {1: 10, 2: 10}), 3: (1, {1: 50, 3: 0}), 4: (3, {3: 30, 4: 30}), 5: (3, {3: 30, 5: 30}), 6: (2, {2: 30, 6: 5})}


TypeError: str() argument 'encoding' must be str, not dict

In [ ]:
# Per match

average_reward_per_match_df = pd.DataFrame.from_dict(average_rewards_per_match, orient="index", columns=["Average Reward"])
average_reward_per_match_df.index.name = "Strategy Name"
average_reward_per_match_df = average_reward_per_match_df.round(3)
# average_reward_per_match_df

In [13]:
strat_names = [str(s) for s in strategies.values()]

score_matrix = pd.DataFrame(index=strat_names, columns=strat_names, dtype=object)

for (s1, s2), scores in duel_rewards.items():
    score_matrix.loc[str(strategies[s2]), str(strategies[s1])] = (scores[s2], scores[s1])
    score_matrix.loc[str(strategies[s1]), str(strategies[s2])] = (scores[s1], scores[s2])

for s in strat_names:
    score_matrix.loc[s, s] = (0, 0)

display(score_matrix)

,Random_Strategy (p_coop=0.5),Random_Strategy (p_coop=0.1),Always_Betray,Always_Cooperate,Patient_Unforgiving (patience=1),Copycat (1st:Cooperate),Periodic (period=1)
Random_Strategy (p_coop=0.5),"(0, 0)","(4, 44)","(6, 26)","(44, 9)","(9, 29)","(25, 25)","(18, 28)"
Random_Strategy (p_coop=0.1),"(44, 4)","(0, 0)","(10, 10)","(50, 0)","(14, 9)","(20, 15)","(28, 8)"
Always_Betray,"(26, 6)","(10, 10)","(0, 0)","(50, 0)","(14, 9)","(14, 9)","(30, 5)"
Always_Cooperate,"(9, 44)","(0, 50)","(0, 50)","(0, 0)","(30, 30)","(30, 30)","(15, 40)"
Patient_Unforgiving (patience=1),"(29, 9)","(9, 14)","(9, 14)","(30, 30)","(0, 0)","(30, 30)","(27, 12)"
Copycat (1st:Cooperate),"(25, 25)","(15, 20)","(9, 14)","(30, 30)","(30, 30)","(0, 0)","(23, 28)"
Periodic (period=1),"(28, 18)","(8, 28)","(5, 30)","(40, 15)","(12, 27)","(28, 23)","(0, 0)"


In [ ]:
victory_matrix = score_matrix.apply(lambda col: col.map(lambda x: int(x[0] > x[1])))
for s in strat_names:
    victory_matrix.loc[s, s] = float("NaN")

In [ ]:
def color_cell(x):
    if not isinstance(x, tuple):
        return ""
    if x[0] == 0 and x[1] == 0:
        return "background-color: black"
    elif x[0] > x[1]:
        return "background-color: green"
    elif x[0] < x[1]:
        return "background-color: darkred"
    else:
        return "background-color: gray"

styled_score_matrix = (
    score_matrix.style
    .map(color_cell)
    .set_properties(**{
        "border": "1px solid black"
    })
    .set_table_styles([
        {"selector": "th", "props": [("border", "2px solid black")]},
        {"selector": "td", "props": [("border", "2px solid black")]}
    ])
)

display(styled_score_matrix)

,Random_Strategy (p_coop=0.5),Random_Strategy (p_coop=0.1),Always_Betray,Always_Cooperate,Patient_Unforgiving (patience=1),Copycat (1st:Cooperate),Periodic (period=1)
Random_Strategy (p_coop=0.5),"(0, 0)","(13, 28)","(8, 18)","(44, 9)","(11, 21)","(24, 19)","(23, 23)"
Random_Strategy (p_coop=0.1),"(28, 13)","(0, 0)","(9, 14)","(46, 6)","(13, 13)","(19, 14)","(30, 5)"
Always_Betray,"(18, 8)","(14, 9)","(0, 0)","(50, 0)","(14, 9)","(14, 9)","(30, 5)"
Always_Cooperate,"(9, 44)","(6, 46)","(0, 50)","(0, 0)","(30, 30)","(30, 30)","(15, 40)"
Patient_Unforgiving (patience=1),"(21, 11)","(13, 13)","(9, 14)","(30, 30)","(0, 0)","(30, 30)","(27, 12)"
Copycat (1st:Cooperate),"(19, 24)","(14, 19)","(9, 14)","(30, 30)","(30, 30)","(0, 0)","(23, 28)"
Periodic (period=1),"(23, 23)","(5, 30)","(5, 30)","(40, 15)","(12, 27)","(28, 23)","(0, 0)"


In [ ]:
import webbrowser
store_styled_matrix_in_html = False
if store_styled_matrix_in_html:
    styled_score_matrix.to_html("styled_score_matrix.html")
    webbrowser.open_new_tab("styled_score_matrix.html")